# 01 · Preparação do ambiente e validação do servidor MCP

Prepara uma sessão do Colab do zero e valida o servidor MCP **sem LLM nenhum**:

| Parte | O que faz | Tempo aproximado |
|---|---|---|
| 1 | Baixa o código do GitHub e copia o recorte do Drive | < 1 min |
| 2 | Instala PostgreSQL 16 e Neo4j 5.26 Community, sem Docker | ~2 min |
| 3 | Carrega o recorte de São Paulo (Inside Airbnb) nos dois bancos | ~4 min |
| 4 | Instala as bibliotecas do servidor MCP num ambiente separado | ~30 s |
| 5 | Lista e chama as ferramentas, o resource e o prompt, inclusive com erros | ~30 s |

**Antes de rodar:** este notebook **não usa GPU**: o runtime padrão, só com CPU, basta e poupa a cota de GPU do Colab. Na Parte 1, ajuste `DRIVE_DIR` para a pasta do Drive com os 8 CSV gerados por `dados/recorte.py`.

In [ ]:
# @title Parte 1 · Código do GitHub e recorte do Drive
import os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/felipeafigueiredo/text-to-sql-and-cypher.git"
REPO_DIR = "/content/text-to-sql-and-cypher"
DRIVE_DIR = "/content/drive/MyDrive/airbnb_recorte"   # pasta do Drive com os 8 CSV do recorte
CACHE_DIR = DRIVE_DIR + "/cache"                     # o .tar.gz do Neo4j fica aqui entre sessões
DADOS = "/content/recorte"
ARQUIVOS = ["neighbourhood.csv", "host.csv", "listing.csv", "listing_amenity.csv", "guest.csv", "review.csv",
            "reviewed.csv", "borders.csv"]
TEMPOS = {}

def etapa(nome, comando):
    # roda um comando de shell mostrando a saída linha a linha e guarda quanto tempo levou
    # (o Colab não mostra o que um subprocesso escreve direto no terminal, por isso o print)
    t = time.time()
    p = subprocess.Popen(comando, shell=True, cwd=REPO_DIR if os.path.isdir(REPO_DIR) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linha in p.stdout:
        print(linha, end="")
    codigo = p.wait()
    TEMPOS[nome] = round(time.time() - t, 1)
    print(f"⏱ {nome}: {TEMPOS[nome]} s")
    if codigo != 0:
        raise RuntimeError(f"'{nome}' falhou (código {codigo}). Veja a saída acima.")

if os.path.isdir(REPO_DIR):
    etapa("atualizar o código", "git pull -q")
else:
    etapa("baixar o código", f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run("git log -1 --format='commit %h · %s'", shell=True, capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount("/content/drive")
faltando = [a for a in ARQUIVOS if not os.path.exists(os.path.join(DRIVE_DIR, a))]
if faltando:
    raise RuntimeError(f"Não achei {faltando} em {DRIVE_DIR}. Ajuste DRIVE_DIR ou suba os CSV.")
os.makedirs(DADOS, exist_ok=True)
for a in ARQUIVOS:
    shutil.copy(os.path.join(DRIVE_DIR, a), os.path.join(DADOS, a))
print(f"✅ {len(ARQUIVOS)} CSV copiados para {DADOS}")

In [ ]:
# @title Parte 2 · Instalar e subir PostgreSQL e Neo4j
etapa("instalar e subir os bancos", f"bash ambiente/instalar_colab.sh bancos '{CACHE_DIR}'")

In [ ]:
# @title Parte 3 · Carregar o recorte nos dois bancos
etapa("bibliotecas da carga", "pip -q install --disable-pip-version-check 'psycopg[binary]==3.3.9' neo4j==6.4.1")
etapa("carregar os bancos", f"python dados/carga.py {DADOS}")

In [ ]:
# @title Parte 4 · Bibliotecas do servidor MCP, num ambiente separado
# O FastMCP 4 usa o mcp 2.x; o adaptador MCP do LangGraph (próximos notebooks) usa o mcp 1.x.
# Por isso o servidor roda com as bibliotecas de servidor_libs/ e o notebook não as importa.
etapa("instalar o servidor", "bash ambiente/instalar_colab.sh servidor")
SERVIDOR_ENV = {**os.environ, "PYTHONPATH": os.path.join(REPO_DIR, "servidor_libs"),
                "AIRBNB_PG_DSN": "host=localhost port=5432 dbname=airbnb user=leitor password=leitor",
                "AIRBNB_NEO4J_URI": "bolt://localhost:7687"}

---
## Parte 5 · Validação sem LLM

Primeiro, o que o servidor declara, pela linha de comando do FastMCP (equivalente ao MCP Inspector). Depois, `servidor/validar.py`: um cliente MCP sobe o servidor por stdio, como um agente faria, e chama cada ferramenta com argumentos certos e errados. Cada caso confere o resultado contra os números conhecidos do recorte ou confere que o erro diz o que corrigir.

In [ ]:
# @title 5.1 · O que o servidor declara: ferramentas, resources e prompt
r = subprocess.run([os.path.join(REPO_DIR, "servidor_libs/bin/fastmcp"), "list", "servidor/server.py",
                    "--resources", "--prompts", "--input-schema"],
                   # o `fastmcp list` sobe o servidor chamando `fastmcp run`, então o bin/ precisa estar no PATH
                   env={**SERVIDOR_ENV, "PATH": os.path.join(REPO_DIR, "servidor_libs/bin") + ":" + os.environ["PATH"],
                        "NO_COLOR": "1", "FASTMCP_SHOW_SERVER_BANNER": "false"},
                   capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])

In [ ]:
# @title 5.2 · Chamar tudo, com acertos e erros
t = time.time()
r = subprocess.run([sys.executable, "servidor/validar.py"], env=SERVIDOR_ENV, capture_output=True, text=True)
TEMPOS["validar o servidor"] = round(time.time() - t, 1)
print(r.stdout[-20000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("Algum caso falhou. Veja os ❌ acima.")

In [ ]:
# @title 5.3 · Resumo da sessão
import platform
print("Python", platform.python_version())
print(subprocess.run("psql --version; /opt/neo4j-server/bin/neo4j --version; echo \"CPUs: $(nproc)\"; free -h | head -n 2",
                     shell=True, capture_output=True, text=True).stdout)
print("\n⏱ tempos (s)")
for k, v in TEMPOS.items():
    print(f"  {k:<30} {v}")
print("\nO relatório completo está em resultados/validacao_servidor.md (painel de arquivos à esquerda).")